# SPARC Edge-Response Analysis (Spacetime Mechanics)

This notebook is a reproducible, transparent workflow to test the **auxiliary edge-response toy model** against SPARC `rotmod` galaxy inputs.

It is designed to work **without numpy/pandas/matplotlib** (stdlib-only). If `matplotlib` is available, it will also generate simple scatter plots.

## What it does
1. Runs the SPARC rotmod runner (`toy_models/sparc_rotmod_runner.py`) on a chosen set of galaxies.
2. Loads the produced `summary.csv` and checks key derived metrics.
3. Runs the correlation analyzer (`toy_models/analyze_summary_correlations.py`) and exports `correlations.csv`.
4. (Optional) Makes quick diagnostic plots.

## Key model idea (one line)
Baryons define a transition scale $R_t$ where $g_{bar}pprox a_0$; an auxiliary field sourced near $R_t$ produces an outer tail $g_{extra}im Q/R$ that can mimic halo-like flat rotation curves.

In [ ]:
import os
import sys

print('Python:', sys.version)
print('Executable:', sys.executable)
print('CWD:', os.getcwd())

## Configure paths and run parameters

- `ROTDMOD_DIR` points at the folder containing `*_rotmod.dat` files.
- `OUT_DIR` is where outputs will be written (`summary.csv` and per-galaxy CSVs).

Notes:
- `max_galaxies=0` means run **all** galaxies in the directory.
- You can vary `sigma_kpc`, `ups_disk`, `ups_bul`, and `a0_ms2` to test sensitivity.

In [ ]:
# Edit these if needed
ROTDMOD_DIR = Path(r'D:\#Documents\#Physics\TPT Paper\Rotmod_LTG')
OUT_DIR = Path(r'toy_models\out_sparc_runs_notebook')

max_galaxies = 25  # set 0 for all galaxies
sigma_kpc = 2.0
ups_disk = 0.5
ups_bul = 0.7
a0_ms2 = 1.2e-10

runner = Path('toy_models') / 'sparc_rotmod_runner.py'
corr_script = Path('toy_models') / 'analyze_summary_correlations.py'

print('ROTDMOD_DIR exists:', ROTDMOD_DIR.exists(), ROTDMOD_DIR)
print('Runner exists:', runner.exists(), runner)
print('Corr script exists:', corr_script.exists(), corr_script)

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
cmd = [
    '--rotmod-dir', str(ROTDMOD_DIR),
    '--sigma-kpc', str(sigma_kpc),
    '--ups-bul', str(ups_bul),
 ]


## Full batch run (timestamped output)


This section runs the SPARC runner on **all** `*_rotmod.dat` files in `ROTDMOD_DIR` (i.e., `max_galaxies=0`) and writes outputs to a timestamped folder under `toy_models/`.



It also exports a `run_config.json` alongside `summary.csv` and `correlations.csv` to make the run fully reproducible.

In [ ]:
import sys
import subprocess
import json
from datetime import datetime
from pathlib import Path

# Make this cell robust if run standalone (defines defaults if earlier cells weren't executed).
try:
    ROTDMOD_DIR
except NameError:
    ROTDMOD_DIR = Path(r"D:\#Documents\#Physics\TPT Paper\Rotmod_LTG")

try:
    sigma_kpc, ups_disk, ups_bul, a0_ms2
except NameError:
    sigma_kpc = 2.0
    ups_disk = 0.5
    ups_bul = 0.7
    a0_ms2 = 1.2e-10

try:
    runner, corr_script
except NameError:
    runner = Path("toy_models") / "sparc_rotmod_runner.py"
    corr_script = Path("toy_models") / "analyze_summary_correlations.py"

run_tag = datetime.now().strftime("%Y%m%d_%H%M%S")
OUT_DIR_FULL = Path("toy_models") / f"out_sparc_runs_full_{run_tag}"
OUT_DIR_FULL.mkdir(parents=True, exist_ok=True)

max_galaxies_full = 0  # 0 means: process all *_rotmod.dat files in ROTDMOD_DIR

cmd = [
    sys.executable, str(runner),
    "--rotmod-dir", str(ROTDMOD_DIR),
    "--out-dir", str(OUT_DIR_FULL),
    "--sigma-kpc", str(sigma_kpc),
    "--ups-disk", str(ups_disk),
    "--ups-bul", str(ups_bul),
    "--a0-ms2", str(a0_ms2),
 ]
if max_galaxies_full and max_galaxies_full > 0:
    cmd += ["--max-galaxies", str(max_galaxies_full)]

print("Running full batch:")
print(" ".join(cmd))
res = subprocess.run(cmd, capture_output=True, text=True)
print(res.stdout)
if res.returncode != 0:
    print("STDERR:\n", res.stderr)
    raise RuntimeError(f"Runner failed with exit code {res.returncode}")

summary_path_full = OUT_DIR_FULL / "summary.csv"
if not summary_path_full.exists():
    raise FileNotFoundError(summary_path_full)

corr_out_full = OUT_DIR_FULL / "correlations.csv"
cmd2 = [sys.executable, str(corr_script), "--summary", str(summary_path_full), "--export", str(corr_out_full)]
print("Running correlations:")
print(" ".join(cmd2))
res2 = subprocess.run(cmd2, capture_output=True, text=True)
print(res2.stdout)
if res2.returncode != 0:
    print("STDERR:\n", res2.stderr)
    raise RuntimeError(f"Correlation script failed with exit code {res2.returncode}")

# Write a machine-readable config file for reproducibility
run_config = {
    "timestamp": run_tag,
    "python": {"version": sys.version, "executable": sys.executable},
    "inputs": {"rotmod_dir": str(ROTDMOD_DIR)},
    "parameters": {
        "sigma_kpc": sigma_kpc,
        "ups_disk": ups_disk,
        "ups_bul": ups_bul,
        "a0_ms2": a0_ms2,
        "max_galaxies": max_galaxies_full,
    },
    "commands": {"runner": cmd, "correlations": cmd2},
    "outputs": {
        "out_dir": str(OUT_DIR_FULL),
        "summary_csv": str(summary_path_full),
        "correlations_csv": str(corr_out_full),
    },
}
config_path = OUT_DIR_FULL / "run_config.json"
config_path.write_text(json.dumps(run_config, indent=2), encoding="utf-8")

print("Done.")
print("Outputs:")
print(" -", OUT_DIR_FULL)
print(" -", summary_path_full)
print(" -", corr_out_full)
print(" -", config_path)